# Notebook 05 — Phase 2A : Zero-Shot Transfer + Amélioration Multi-Sources
**Projet: SentimentsLoupe**

---
## Objectif

1. **Scraper** Trustpilot (9 sites e-commerce) + Jumia Maroc (mlp-anniversaire)
2. **Zero-Shot** : Tester le champion Phase 1 (entraîné sur Flipkart) sur ces nouvelles sources
3. **Améliorer** : Ré-entraîner avec Trustpilot, tester sur Jumia
4. **Comparer les gaps** : Domaine vs Langue → justifier Phase 2B (XLM-RoBERTa)

---
## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.metrics import f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression

from src.config.settings import (
    MODELS_DIR, PLOTS_DIR, RAW_DIR, PROCESSED_DIR, RANDOM_STATE
)
from src.preprocessing.source_normalizer import SourceNormalizer
from src.preprocessing.feature_engineering import FeatureEngineer
from src.data_collection.trustpilot_scraper import TrustpilotScraper
from src.data_collection.jumia_fix_scraper import JumiaListingScraper

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('Set2')

print('Setup OK')

In [ ]:
# Charger les artefacts Phase 1
champion_obj = joblib.load(str(MODELS_DIR / 'champion_model.pkl'))
champion_model = champion_obj['model']
champion_has_svd = champion_obj.get('svd', None)

feat_eng_p1 = joblib.load(str(MODELS_DIR / 'feature_engineer.pkl'))

with open(str(MODELS_DIR / 'phase1_metrics.json'), 'r') as f:
    metrics_p1 = json.load(f)

F1_REF = metrics_p1['champion_metrics']['f1_macro']
ACC_REF = metrics_p1['champion_metrics']['accuracy']

print(f'Champion Phase 1: [{metrics_p1["best_strategy"]}] {metrics_p1["best_model"]}')
print(f'F1-Macro référence (Flipkart): {F1_REF:.4f}')
print(f'Accuracy référence (Flipkart):  {ACC_REF:.4f}')

---
## 1. Source A — Trustpilot Multi-Domaines (Scraping)

On scrappe **9 sites e-commerce** (Amazon, Apple, Samsung, etc.) ~7 pages chacun.

In [ ]:
TRUSTPILOT_PARQUET = RAW_DIR / 'trustpilot_multidomain.parquet'
TRUSTPILOT_CSV = RAW_DIR / 'trustpilot_multidomain.csv'

BEST_DOMAINS = [
    ("www.amazon.com", 7),
    ("www.bestbuy.com", 7),
    ("www.walmart.com", 7),
    ("www.target.com", 7),
    ("www.apple.com", 7),
    ("www.samsung.com", "/uk", 7),
    ("www.mi.com", 7),
    ("www.oneplus.com", 7),
    ("www.aliexpress.com", 7),
]

if TRUSTPILOT_PARQUET.exists():
    print('Fichier parquet existant trouvé. Chargement...')
    df_trustpilot = pd.read_parquet(TRUSTPILOT_PARQUET)
    df_trustpilot.to_csv(TRUSTPILOT_CSV, index=False)
    print(f'Chargé: {len(df_trustpilot)} avis depuis {TRUSTPILOT_PARQUET.name}')
elif TRUSTPILOT_CSV.exists():
    print('Fichier CSV existant trouvé. Chargement...')
    df_trustpilot = pd.read_csv(TRUSTPILOT_CSV)
    print(f'Chargé: {len(df_trustpilot)} avis depuis {TRUSTPILOT_CSV.name}')
else:
    print('Lancement du scraping (9 sites × 7 pages)...')
    all_trustpilot = []
    for domain_info in BEST_DOMAINS:
        if len(domain_info) == 3:
            domain, path, pages = domain_info
        else:
            domain, pages = domain_info
            path = ""
        print(f"\n[TRUSTPILOT] Scraping {domain}{path} ({pages} pages)...")
        scraper = TrustpilotScraper(domain=domain, path=path, max_pages=pages)
        df = scraper.scrape()
        if not df.empty:
            all_trustpilot.append(df)
            partial_path = RAW_DIR / f'trustpilot_{domain.replace(".","_")}.csv'
            df.to_csv(partial_path, index=False)
    if all_trustpilot:
        df_trustpilot = pd.concat(all_trustpilot, ignore_index=True)
        df_trustpilot.to_csv(TRUSTPILOT_CSV, index=False)
    else:
        df_trustpilot = pd.DataFrame()

print(f'\n=== TOTAL Trustpilot: {len(df_trustpilot)} avis ===')
if not df_trustpilot.empty:
    print(f'Domaines: {df_trustpilot["domain"].unique()}')
    df_trustpilot[['text', 'rating', 'domain']].head(10)

---
## 2. Source B — Jumia Maroc (Scraping)

Jumia.ma = le Amazon marocain. Avis en Français et Darija romanisé.

In [ ]:
JUMIA_RAW = RAW_DIR / 'jumia_maroc.parquet'

if JUMIA_RAW.exists():
    print(f'Fichier existant trouvé. Chargement...')
    df_jumia = pd.read_parquet(JUMIA_RAW)
    print(f'Chargé: {len(df_jumia)} avis')
else:
    print("Scraping Jumia page mlp-anniversaire...")
    scraper = JumiaListingScraper(
        "https://www.jumia.ma/mlp-anniversaire/",
        max_products=30
    )
    df_jumia = scraper.scrape()
    if not df_jumia.empty:
        df_jumia.to_parquet(JUMIA_RAW, index=False)
        print(f'\u2705 Sauvegarde: {JUMIA_RAW} ({len(df_jumia)} avis)')
    else:
        df_jumia = pd.DataFrame()

print(f'Jumia: {len(df_jumia)} avis collectés')
if not df_jumia.empty:
    print(f'Shape: {df_jumia.shape}')
    df_jumia[['text', 'rating', 'product_name']].head(10)

---
## 3. Normalisation Inter-Sources

On applique le pipeline Phase 1 (TextCleaner + mapping rating → sentiment → label)
à toutes les sources. Le `SourceNormalizer` crée un schéma unifié.

In [ ]:
# Afficher l'impact du TextCleaner sur un exemple Jumia
if not df_jumia.empty:
    print("\n🔍 Impact du TextCleaner sur un avis Jumia:")
    print("=" * 60)
    
    # Chercher un avis avec accents/arabe
    sample = df_jumia.iloc[2]['text'] if len(df_jumia) > 2 else df_jumia.iloc[0]['text']
    print(f"\n📝 Original:    {sample}")
    
    from src.preprocessing.text_cleaner import TextCleaner
    cleaner = TextCleaner(lemmatize=False, remove_stopwords=False)
    cleaned = cleaner.clean(sample)
    print(f"🧹 Après clean: {cleaned}")
    print(f"\n⚠️ Problème: les accents (é, à, ü) et caractères arabes sont supprimés")

In [ ]:
normalizer = SourceNormalizer()
sources = {}

# Source 1: Flipkart (référence Phase 1)
df_flipkart = pd.read_csv(str(PROCESSED_DIR / 'flipkart_cleaned.csv'))
df_flipkart['source'] = 'flipkart'
df_flipkart['true_label'] = df_flipkart['sentiment']
sources['flipkart'] = df_flipkart
print(f'flipkart: {len(df_flipkart)} avis (référence)')

# Source 2: Trustpilot
if not df_trustpilot.empty:
    df_tp_norm = normalizer.normalize(df_trustpilot, source='trustpilot')
    sources['trustpilot'] = df_tp_norm
    print(f'trustpilot: {len(df_tp_norm)} avis normalisés')

# Source 3: Jumia
if not df_jumia.empty:
    df_jm_norm = normalizer.normalize(df_jumia, source='jumia')
    sources['jumia'] = df_jm_norm
    print(f'jumia: {len(df_jm_norm)} avis normalisés')

# Combiner
df_all = pd.concat(sources.values(), ignore_index=True)
print(f'\nTotal combiné: {len(df_all)} avis')

In [ ]:
# Tableau récapitulatif par source
summary = normalizer.summary(df_all)
summary

---
## 4. Zero-Shot Evaluation — Champion Phase 1

On teste le champion Flipkart sur Trustpilot (même langue, domaine différent) et Jumia (langue différente) **sans ré-entraînement**.

In [ ]:
results_zs = []

for name, df_norm in sources.items():
    if name == 'flipkart':
        f1 = F1_REF
        acc = ACC_REF
    else:
        X = feat_eng_p1.transform(df_norm)
        if champion_has_svd is not None:
            X = champion_has_svd.transform(X)
        
        y_true = df_norm['label'].values
        y_pred = champion_model.predict(X)
        
        mask = y_true != -1
        if mask.any():
            f1 = f1_score(y_true[mask], y_pred[mask], average='macro')
            acc = accuracy_score(y_true[mask], y_pred[mask])
        else:
            f1, acc = 0.0, 0.0
    
    gap = F1_REF - f1
    results_zs.append({
        'source': name,
        'f1_macro': round(f1, 4),
        'accuracy': round(acc, 4),
        'n_samples': len(df_norm),
        'gap_vs_flipkart': round(gap, 4)
    })

df_results_zs = pd.DataFrame(results_zs)
print("\n📊 Zero-Shot Evaluation — Champion Phase 1")
print("=" * 60)
df_results_zs

### Interprétation des résultats Zero-Shot

| Test | F1-Macro | Gap vs Flipkart | Interprétation |
|------|----------|----------------|----------------|
| **Flipkart** (réf) | 0.6491 | 0 | Modèle de référence |
| **Trustpilot** | ~0.43 | ~0.22 | **Gap Domaine** : même langue, site différent |
| **Jumia** | ~0.34 | ~0.31 | **Gap Langue** : FR/Darija vs Anglais |

→ Le gap LANGUE est plus grand que le gap DOMAINE

---
## 5. Amélioration — Ré-entraînement avec Trustpilot

On fusionne Flipkart (7694) + Trustpilot (1235) = **8929 avis** pour créer un meilleur modèle.

In [ ]:
# Fusionner Flipkart + Trustpilot
df_train = pd.concat([sources['flipkart'], sources['trustpilot']], ignore_index=True)
print(f'Dataset entraînement: {len(df_train)} avis')
print(f'  Flipkart: {len(sources["flipkart"])}')
print(f'  Trustpilot: {len(sources["trustpilot"])}')
print(f'\nDistribution des sentiments:')
print(df_train['sentiment'].value_counts().to_string())

In [ ]:
# Filtrer les lignes avec label invalide (NaN) ou review_clean vide
df_train_clean = df_train.dropna(subset=['label']).copy()
df_train_clean = df_train_clean[df_train_clean['review_clean'].str.strip() != '']
print(f'Lignes valides pour entraînement: {len(df_train_clean)}/{len(df_train)}')
print(f'Lignes supprimées (label NaN ou texte vide): {len(df_train) - len(df_train_clean)}')

# Créer un NOUVEAU FeatureEngineer (apprend le vocabulaire FK+TP, pas de features numériques)
feat_eng_v2 = FeatureEngineer(use_numeric_features=False)
X_train = feat_eng_v2.fit_transform(df_train_clean)
y_train = df_train_clean['label'].values

# Entraîner LogisticRegression (même config que champion Phase 1)
model_v2 = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=2000,
    random_state=42,
    n_jobs=-1
)
model_v2.fit(X_train, y_train)

# Évaluation sur train
y_pred_train = model_v2.predict(X_train)
f1_train = f1_score(y_train, y_pred_train, average='macro')
acc_train = accuracy_score(y_train, y_pred_train)

print(f'Modèle v2 entraîné: LogisticRegression (class_weight=balanced)')
print(f'Features: {X_train.shape[1]}')
print(f'\nF1-Macro (train): {f1_train:.4f}')
print(f'Accuracy (train):  {acc_train:.4f}')

---
## 6. Test du nouveau modèle sur Trustpilot + Jumia

On teste le modèle v2 (FK+TP) sur les deux sources pour mesurer l'impact du ré-entraînement.

In [ ]:
# ── Test sur Trustpilot ──
df_tp_test = sources['trustpilot'].copy()
X_tp = feat_eng_v2.transform(df_tp_test)
y_tp_true = df_tp_test['label'].values
y_tp_pred = model_v2.predict(X_tp)

mask_tp = y_tp_true != -1
if mask_tp.any():
    f1_tp_v2 = f1_score(y_tp_true[mask_tp], y_tp_pred[mask_tp], average='macro')
    acc_tp_v2 = accuracy_score(y_tp_true[mask_tp], y_tp_pred[mask_tp])
else:
    f1_tp_v2, acc_tp_v2 = 0.0, 0.0

print(f'📊 Test du modèle v2 (Flipkart+Trustpilot) sur Trustpilot')
print("=" * 50)
print(f'F1-Macro: {f1_tp_v2:.4f}')
print(f'Accuracy: {acc_tp_v2:.4f}')
f1_tp_p1_zs = df_results_zs[df_results_zs["source"]=="trustpilot"]["f1_macro"].values[0]
print(f'\nComparaison avec Phase 1 (champion seul):')
print(f'  Phase 1: F1 = {f1_tp_p1_zs:.4f}')
print(f'  Phase 2: F1 = {f1_tp_v2:.4f}')
print(f'  Amélioration: {f1_tp_v2 - f1_tp_p1_zs:.4f}')

# ── Test sur Jumia ──
df_jm_test = sources['jumia']
X_jm = feat_eng_v2.transform(df_jm_test)
y_jm_true = df_jm_test['label'].values
y_jm_pred = model_v2.predict(X_jm)

mask = y_jm_true != -1
if mask.any():
    f1_jm_v2 = f1_score(y_jm_true[mask], y_jm_pred[mask], average='macro')
    acc_jm_v2 = accuracy_score(y_jm_true[mask], y_jm_pred[mask])
else:
    f1_jm_v2, acc_jm_v2 = 0.0, 0.0

print(f'\n📊 Test du modèle v2 (Flipkart+Trustpilot) sur Jumia')
print("=" * 50)
print(f'F1-Macro: {f1_jm_v2:.4f}')
print(f'Accuracy: {acc_jm_v2:.4f}')
f1_jm_p1_zs = df_results_zs[df_results_zs["source"]=="jumia"]["f1_macro"].values[0]
print(f'\nComparaison avec Phase 1 (champion seul):')
print(f'  Phase 1: F1 = {f1_jm_p1_zs:.4f}')
print(f'  Phase 2: F1 = {f1_jm_v2:.4f}')
print(f'  Amélioration: {f1_jm_v2 - f1_jm_p1_zs:.4f}')

---
## 7. Analyse d'Erreurs Jumia

Regardons concrètement pourquoi le modèle échoue sur les avis Jumia.

In [ ]:
# Analyse des erreurs du meilleur modèle (v2) sur Jumia
df_jm_errors = df_jm_test.copy()
df_jm_errors['pred_label'] = y_jm_pred
df_jm_errors['pred_sentiment'] = df_jm_errors['pred_label'].map({0: 'négatif', 1: 'neutre', 2: 'positif'})
df_jm_errors['correct'] = df_jm_errors['label'] == df_jm_errors['pred_label']

errors = df_jm_errors[~df_jm_errors['correct']]
correct = df_jm_errors[df_jm_errors['correct']]

print(f'✅ {len(correct)}/{len(df_jm_errors)} avis corrects')
print(f'❌ {len(errors)}/{len(df_jm_errors)} avis mal classés\n')

print("📋 Exemples d'erreurs:")
print("=" * 80)
for i, (_, row) in enumerate(errors.head(6).iterrows()):
    print(f'\n{i+1}. Review: "{row["review"][:100]}"')
    print(f'   Rating: {int(row["rating"])}/5 → Vrai: {row["sentiment"]}')
    print(f'   Prédit: {row["pred_sentiment"]}')
    # Explication
    if any(c in row['review'] for c in 'éèêëàâùûôîïç'):
        print(f'   🔴 Cause: Accents français supprimés par TextCleaner')
    if any(ord(c) > 0x0600 and ord(c) < 0x06FF for c in row['review']):
        print(f'   🔴 Cause: Arabe/Darija totalement supprimé par TextCleaner')
    print("   " + "-" * 50)

---
## 8. Comparaison Finale — Tableau des Gaps

Tableau récapitulatif pour la soutenance.

In [ ]:
# Récupérer les valeurs
f1_tp_p1 = df_results_zs[df_results_zs['source']=='trustpilot']['f1_macro'].values[0]
f1_jm_p1 = df_results_zs[df_results_zs['source']=='jumia']['f1_macro'].values[0]

# Créer le tableau final (avec les valeurs de f1_tp_v2 et f1_jm_v2 calculées dans la section 6)
df_final = pd.DataFrame({
    'Modèle': [
        'Phase 1 (Flipkart seul)',
        'Phase 2A (Flipkart+Trustpilot)',
    ],
    'F1 Trustpilot': [
        round(f1_tp_p1, 4),
        round(f1_tp_v2, 4),
    ],
    'F1 Jumia (FR/Darija)': [
        round(f1_jm_p1, 4),
        round(f1_jm_v2, 4),
    ],
    'Gap Domaine': [
        round(F1_REF - f1_tp_p1, 4),
        round(F1_REF - f1_tp_v2, 4),
    ],
    'Gap Langue': [
        round(F1_REF - f1_jm_p1, 4),
        round(F1_REF - f1_jm_v2, 4),
    ],
})

print("\n🏆 TABLEAU COMPARATIF FINAL")
print("=" * 70)
df_final

In [ ]:
# Barplot comparatif
fig, ax = plt.subplots(figsize=(10, 6))

sources_names = ['Flipkart\n(référence)', 'Trustpilot\n(Phase 1)', 'Jumia\n(Phase 1)', 'Jumia\n(Phase 2A)']
f1_values = [F1_REF, f1_tp_p1, f1_jm_p1, f1_jm_v2]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']

bars = ax.bar(sources_names, f1_values, color=colors, edgecolor='white', linewidth=1.5)

# Ajouter les valeurs sur les barres
for bar, v in zip(bars, f1_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
            f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')

# Ajouter les gaps
gap_domaine = F1_REF - f1_tp_p1
gap_langue = F1_REF - f1_jm_p1

ax.annotate(f'Gap Domaine\n{round(gap_domaine,4)}', xy=(1, F1_REF), xytext=(1, f1_tp_p1),
            arrowprops=dict(arrowstyle='<->', color='gray', lw=2), fontsize=9, color='gray')
ax.annotate(f'Gap Langue\n{round(gap_langue,4)}', xy=(2, F1_REF), xytext=(2, f1_jm_p1),
            arrowprops=dict(arrowstyle='<->', color='red', lw=2), fontsize=9, color='red')

# Amélioration
ax.annotate(f'+{round(f1_jm_v2-f1_jm_p1,4)}', xy=(3, f1_jm_v2), xytext=(2.35, (f1_jm_p1+f1_jm_v2)/2),
            arrowprops=dict(arrowstyle='->', color='green', lw=2), fontsize=10, color='green', fontweight='bold')

ax.set_title('Comparaison des F1-Macro : Gap Domaine vs Gap Langue', fontsize=14, fontweight='bold')
ax.set_ylabel('F1-Macro')
ax.set_ylim(0, 0.75)
ax.axhline(y=F1_REF, color='green', linestyle='--', linewidth=1.2, alpha=0.7,
           label=f'Référence Flipkart ({F1_REF:.4f})')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '05_comparaison_gaps.png'), bbox_inches='tight', dpi=150)
plt.show()
print(f'✅ Graphique sauvegardé: {PLOTS_DIR / "05_comparaison_gaps.png"}')

---
## 9. Conclusion et Interprétation

### Résumé des Gaps

| Type de Gap | Valeur | Interprétation |
|-------------|--------|----------------|
| **Gap Domaine** (Flipkart → Trustpilot) | ~0.22 | Même langue (anglais), site différent |
| **Gap Langue** (Flipkart → Jumia Phase 1) | ~0.31 | Langue différente (FR/Darija) |
| **Gap Langue** (Flipkart → Jumia Phase 2A) | ~0.27 | Après ajout Trustpilot |
| **Amélioration** (Phase 1 → Phase 2A sur Jumia) | ~+0.04 | Très faible progrès |

### Conclusions pour la Soutenance

1. **Le changement de LANGUE** est bien plus pénalisant que le changement de domaine
2. **L'ajout de données anglaises** (Trustpilot) n'a quasi pas amélioré les performances sur Jumia
3. **Le TextCleaner actuel supprime les accents et l'arabe** → information perdue pour le modèle
4. **Le TF-IDF anglais est incapable** de capturer le vocabulaire darija/français

### Prochaine étape : Phase 2B

Solution : **XLM-RoBERTa** (modèle multilingue pré-entraîné) qui comprend le français, l'arabe,
et peut être fine-tuné sur le darija. Voir Notebook 06.